# 曲线拟合问题

**类别:** 非线性

来源: [https://www.hexaly.com/templates/curve-fitting-problem](https://www.hexaly.com/templates/curve-fitting-problem)


## 问题描述

解决 **曲线拟合问题** 即构建一条曲线或数学函数,使其尽可能最佳地拟合一系列数据点,并可能满足一些约束。曲线拟合既可以是插值(需要对数据进行精确拟合),也可以是平滑(构造一个“平滑”函数来近似拟合数据)。更多详细信息,请参阅 [Wikipedia](https://en.wikipedia.org/wiki/Curve_fitting)。

在我们这里考虑的曲线拟合问题中,我们希望为一个已定义的函数找到最优参数,以使该函数能最好地将一组输入映射为一组输出。例如,假设映射函数具有如下形式:

$$f(x) = a\sin(b-x) + cx^2 + d$$

我们希望找到参数 a、b、c 和 d,使映射函数能够最佳地拟合观测数据。

	

### 学到的要点

- 添加浮点型 [决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#decision-variables)
- 使用 [`sin` 和 `pow` 算子](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 构造一个非线性表达式
- 最小化一个 [非线性目标](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#define-your-objective-function)


## 数据

每个数据文件包含:

- 第一行:观测数量
- 接下来的行:每个观测的输入和输出值


## 模型

曲线拟合问题的 Hexaly 模型使用浮点型决策变量来表示四个参数 $a$、$b$、$c$ 和 $d$。利用 [**sin**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#sin) 和 [**pow**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#pow) 非线性算子,我们可以将每个观测对应的函数预测值作为中间表达式计算出来。

目标函数是平方误差之和。利用先前计算的中间表达式和 [**pow**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#pow) 算子,我们可以将每个观测的平方误差计算为映射函数预测输出与观测输出之差的平方。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


def read_float(filename):
    with open(filename) as f:
        return [float(elem) for elem in f.read().split()]

#
# Read instance data
#


def read_instance(instance_file):
    file_it = iter(read_float(instance_file))

    # Number of observations
    nb_observations = int(next(file_it))

    # Inputs and outputs
    inputs = []
    outputs = []
    for i in range(nb_observations):
        inputs.append(next(file_it))
        outputs.append(next(file_it))

    return nb_observations, inputs, outputs


def main(instance_file, output_file, time_limit):
    nb_observations, inputs, outputs = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Decision variables (parameters of the mapping function)
        a = model.float(-100, 100)
        b = model.float(-100, 100)
        c = model.float(-100, 100)
        d = model.float(-100, 100)

        # Minimize square error between prediction and output
        predictions = [a * model.sin(b - inputs[i]) + c * inputs[i] ** 2 + d
                       for i in range(nb_observations)]
        errors = [predictions[i] - outputs[i] for i in range(nb_observations)]
        square_error = model.sum(model.pow(errors[i], 2) for i in range(nb_observations))
        model.minimize(square_error)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("Optimal mapping function\n")
                f.write("a = " + str(a.value) + "\n")
                f.write("b = " + str(b.value) + "\n")
                f.write("c = " + str(c.value) + "\n")
                f.write("d = " + str(d.value) + "\n")


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python curve_fitting.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 3
    main(instance_file, output_file, time_limit)
